# RESULTADOS OBTIDOS

### **Weighted Accuracy**

----------------------------------------------------------------------------------------------------
#### Modelo base line:

**matriz de confusão**

RESULTADOS: VALIDAÇÃO (BASELINE) | Estratégia: MEAN

- WA_murmur: 0.6778

                     | True Present |True Unknown | True Absent
        Pred Present |          27  |          4  |        62
        Pred Unknown |           0  |          6  |        13
        Pred Absent  |           0  |          0  |        30

----------------------------------------------------------------------------------------------------
#### Modelo melhorado:

**matriz de confusão**

RESULTADOS: VALIDAÇÃO (IMPROVED) | Estratégia: TOP-K E PESOS SUAVIZADOS

- WA_murmur: 0.8296

                    True Present  True Unknown  True Absent
        Pred Present            27             5           18
        Pred Unknown             0             1            1
        Pred Absent              0             4           86



----------------------------------------------------------------------------------------------------
RESULTADOS: VALIDAÇÃO (RESNET50) | Estratégia: TOPK + SUAVIZAÇÃO DOS PESOS

- **WA_murmur: 0.6259**

                     True Present  True Unknown  True Absent
        Pred Present            14             4            4
        Pred Unknown             0             0            2
        Pred Absent             13             6           99

        --- APÓS CALIBRAÇÃO DE LIMIARES (0.31, 0.35) ---

- **WA_murmur Otimizado: 0.7111**

                    True Present  True Unknown  True Absent
        Pred Present            20             6           13
        Pred Unknown             0             1            3
        Pred Absent              7             3           89

----------------------------------------------------------------------------------------------------

## INTERPRETAÇÃO E ANÁLISE DOS DADOS

### Modelo baseline:

**Desenho da estratégia de treino**

Foi desenhada para resolver os dois maiores desafios deste problema (a deteção de sopros cardíacos da competição CirCor DigiScope): o desbalanceamento severo das classes e o desfasamento entre a unidade de entrada do modelo (janela de áudio) e a unidade de diagnóstico clínico (paciente).

- Entrada do Modelo (Janelas): Os áudios longos dos pacientes foram segmentados no pré-processamento em janelas curtas de espectrograma log-mel (com dimensão [frames, n_mels, 1]). A CNN 2D aprende características locais (tempo-frequência) analisando cada janela isolada.  

- Agregação por Paciente (to_patient): Clinicamente, não interessa se uma janela teve ruído; interessa o diagnóstico do indivíduo. Por isso, o script agrupa todas as janelas que partilham o mesmo pid e calcula a média das probabilidades previstas para esse doente:

    $$\bar{P}(\text{classe} \mid \text{paciente}) = \frac{1}{K} \sum_{k=1}^K P(\text{classe} \mid \text{janela}_k)$$

- Compensação de Classes Desbalanceadas no Treino (class_weight): Na base de dados, a distribuição dos pacientes está fortemente inclinada para a ausência de sopro:   Absent: $\approx 73.8\%$   Present: $\approx 19.0\%$   Unknown: $\approx 7.2\%$   Se a rede treinasse sem ajuste, tenderia a prever sempre Absent para minimizar o erro global. Para contrariar isso, o script calcula pesos de classe inversamente proporcionais às contagens do conjunto de treino:
    $$\text{weight}_i = \frac{N_{\text{total}}}{3 \times N_i}$$

**Otimização Orientada à Métrica da Competição (Weighted Accuracy - WA)** : A métrica oficial da competição PhysioNet atribui recompensas desiguais aos acertos com base no vetor de pesos METRIC_W = [5.0, 3.0, 1.0]:  
 -  Present: peso $5.0$ |  Unknown: peso $3.0$ |  Absent: peso $1.0$


 Esta otimização é implementada pelas:

- Regra de Decisão do Doente: No código original, para decidir a classe final, as probabilidades médias eram ponderadas por este vetor (score = pr * METRIC_W) antes de tirar o argmax;

- Callback Customizado (PatientWA): Em cada final de época, a função on_epoch_end corre a validação completa, agrega por paciente e calcula o WA real da validação (val_wa), permitindo monitorizar a métrica que verdadeiramente importa e não apenas a acurácia por janela;

**Gestão Dinâmica da Convergência (Callbacks)**: O treino utiliza dois mecanismos automáticos para evitar overfitting e garantir que a rede converge de forma estável:   
- ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3): Se o erro de validação (val_loss) não diminuir durante 3 épocas consecutivas, a taxa de aprendizagem (learning rate) é reduzida para metade (passando de $0.001$ para $0.0005$, depois $0.00025$, etc.), permitindo que a descida do gradiente refine os pesos.   
- EarlyStopping(monitor="val_wa", mode="max", patience=8, restore_best_weights=True): Vigia diretamente o val_wa ao nível do paciente. Se o modelo passar 8 épocas sem conseguir superar o seu recorde de WA, o treino é interrompido e os pesos da melhor época são automaticamente restaurados para o ficheiro final.

Isto está definido no ficheiro cnn_structured.py

## **RESULTADOS**


**=== VALIDAÇÃO (nível de paciente) ===**

- **A_murmur: 0.6778**


                    |True Present | True Unknown | True Absent
        Pred Present  |          27 |          4   |       62
        Pred Unknown  |          0  |          6   |       13
        Pred Absent   |          0  |          0   |       30 

- O modelo está a prever todos os casos como murmur present, o que não é o ideal, porque há imensos falos postivos, ou deja numa situação clinica real, demasiados pacientes saudáveis seriam classificados como doentes. Mas a WA é elevada devido ao facto de dar-mos mais peso à classificação correta com present do que as restantes (5), o que faz com que esta métrica suba. 
- --> é demasiado agressivo !!

#### Como podemos melhorar estes resultados??

- **Calibração dos limiares ótimos de decisão**: Em vez de usar a regra rígida atual (argmax sobre as probabilidades multiplicadas pelos pesos 5:3:1, onde basta $\bar{p}_{\text{present}} \ge 0.17$ para prever sopro), otimizar diretamente os limiares na validação para converter probabilidades contínuas em classes e maximizar a métrica oficial ($\text{WA}$).

- **Suavização dos pesos das classes no treino (Smoothed Class Weights)**: Substituir os pesos agressivos pela raiz quadrada da frequência inversa das classes (normalizada para média $1.0$), permitindo que a rede calibre probabilidades mais realistas sem ficar demasiado polarizada, mantendo a ponderação 5:3:1 apenas na avaliação final.

- **Agregação por Top-$K$ janelas por paciente**: Trocar a média simples de todas as janelas pela média das $K$ janelas com maior probabilidade de sopro (Present), evitando que um sopro nítido em poucas gravações/janelas seja diluído por várias janelas sem sinal audível.

## Modelo Improved

**Aplicação de todas as soluções indicadas em cima.**

- WA_murmur: 0.8296

                    True Present  True Unknown  True Absent
        Pred Present            27             5           18
        Pred Unknown             0             1            1
        Pred Absent              0             4           86

- Com a agregação Top-3 e os pesos suavizados, a CNN atingiu um $\text{WA}_{\text{murmur}}$ de 0.8296 na validação. 
- A sensibilidade de 100% na classe Present (27 em 27 pacientes detetados, sem qualquer falso negativo), e matém uma especificidade de 81,9% nos pacientes saudáveis (Absent). 
- A nivel de falsos postivos, houve apenas 18 pessoas sauáveis que foram classificadas como murmur present. Isto tem um custo baixo, tanto que nos pesoas do enunciado conta como 1, ou seja o seu custo a nivel clinico é baixo comparando com um falso negativo !





## Comparação com ResNet 50

**Aplicação de todas as soluções indicadas em cima + congelamento da base da resnet, deixando apenas a head dense, que tem cerca de 131 331 parametros treináveis.**

- **WA_murmur: 0.6259**

                     True Present  True Unknown  True Absent
        Pred Present            14             4            4
        Pred Unknown             0             0            2
        Pred Absent             13             6           99

        --- APÓS CALIBRAÇÃO DE LIMIARES (0.31, 0.35) ---

- **WA_murmur Otimizado: 0.7111**

                    True Present  True Unknown  True Absent
        Pred Present            20             6           13
        Pred Unknown             0             1            3
        Pred Absent              7             3           89

- Para avaliar o desempenho do Transfer Learning com uma arquitetura pré-treinada de grande escala, implementámos a ResNet50 com os pesos do ImageNet congelados e uma cabeça densa treinável (~131 mil parâmetros), mantendo exatamente a mesma agregação Top-3 e a suavização dos pesos das classes usada na nossa CNN personalizada.Na avaliação inicial na validação (com a regra de decisão padrão argmax), o modelo atingiu um $\text{WA}_{\text{murmur}}$ de 0,6259.

- A matriz de confusão revelou um comportamento excessivamente conservador: embora identificasse corretamente 99 dos 105 pacientes saudáveis (Absent), falhou quase metade dos casos clínicos reais, detetando apenas 14 dos 27 pacientes com sopro (Present) e deixando passar 13 falsos negativos — o que penalizou fortemente a métrica devido ao peso elevado (5) atribuído a esta classe.

- Ao aplicarmos a calibração dos limiares de decisão na validação ($\theta_{\text{Present}} = 0,31$ e $\theta_{\text{Unknown}} = 0,35$), o $\text{WA}_{\text{murmur}}$ otimizado subiu para 0,7111 (+8,52 pontos percentuais). Este ajuste permitiu recuperar 6 pacientes com sopro que estavam a ser descartados (subindo a sensibilidade na classe Present de 51,9% para 74,1%, com 20 em 27 casos detetados) e identificar 1 caso Unknown, à custa de um aumento ligeiro nos falsos positivos (de 4 para 13). 

- Ainda assim, mesmo após a calibração, a ResNet50 ficou claramente abaixo da nossa CNN 2D desenhada de raiz (0,8296), confirmando que os filtros pré-treinados em fotografias naturais sofrem de incompatibilidade de domínio (domain shift) quando aplicados diretamente a espectrogramas acústicos do coração.
